In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tab_transformer_pytorch import TabTransformer, FTTransformer
from preprocessing import get_features_and_target
from sklearn.preprocessing import LabelEncoder
from RMSELoss import RMSELoss
import plotly.graph_objects as go

# Get Dataset

In [2]:
train_df = pd.read_csv("data/train_data.csv")
dev_df = pd.read_csv("data/development_data.csv")

target_column = "PullTest (N)"  

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)


# Augment for New Columns

In [7]:
# Augment the training DataFrame with empty columns for calculations

train_df_c = train_df
train_df_c['FPull_4d_310MPa'] = ''
train_df_c['FPull_5d_310MPa'] = ''
train_df_c['FPull_4d_365MPa'] = ''
train_df_c['FPull_5d_365MPa'] = ''
train_df_c['FPull_4d_440MPa'] = ''
train_df_c['FPull_5d_440MPa'] = ''
train_df_c['FPull_4d_sig_06_365MPa'] = ''
train_df_c['FPull_4d_sig_07_365MPa'] = ''
train_df_c['FPull_4d06_365MPa_nosquare'] = ''
train_df_c['FPull_4d07_365MPa_nosquare'] = ''

# Calculate values

Formula: 

Shear-failure model: 

F = A * τ 

A = pi * d² / 4

τ ≈ 0.8 * σ

Nugget Diameter Relation:

d ≈ k * √t           (with k = 4 or 5)

# F = (pi / 4) * ((4 or 5) * √t)² * (0.8 * σ)

In [12]:
for i in range(train_df_c.shape[0]):

    # Calculate the thickness based on the minimum of either Thickness A or Thickness B
    if train_df_c['Thickness A (mm)'][i] <= train_df_c['Thickness B (mm)'][i]:
        t = train_df_c['Thickness A (mm)'][i]
    else:
        t = train_df_c['Thickness B (mm)'][i]

    # Calculate the pull force for different diameters and yield strengths
    f_pull_4d_310MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.8 * 310)
    train_df_c.loc[i, 'FPull_4d_310MPa'] = round(f_pull_4d_310MPa, 1)
    f_pull_5d_310MPa = (np.pi/4) * np.square(5 * np.sqrt(t)) * (0.8 * 310)
    train_df_c.loc[i, 'FPull_5d_310MPa'] = round(f_pull_5d_310MPa, 1)

    f_pull_4d_365MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.7 * 365)
    train_df_c.loc[i, 'FPull_4d_365MPa'] = round(f_pull_4d_365MPa, 1)
    f_pull_5d_365MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.8 * 365)
    train_df_c.loc[i, 'FPull_5d_365MPa'] = round(f_pull_5d_365MPa, 1)

    f_pull_4d06_365MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.6 * 365)
    train_df_c.loc[i, 'FPull_4d_sig_06_365MPa'] = round(f_pull_4d06_365MPa, 1)
    f_pull_5d07_365MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.7 * 365)
    train_df_c.loc[i, 'FPull_4d_sig_07_365MPa'] = round(f_pull_5d07_365MPa, 1)

    f_pull_4d06_365MPa_nosquare = (np.pi/4) * np.square(4 * t) * (0.6 * 365)
    train_df_c.loc[i, 'FPull_4d06_365MPa_nosquare'] = round(f_pull_4d06_365MPa_nosquare, 1)
    f_pull_5d07_365MPa_nosquare = (np.pi/4) * np.square(4 * t) * (0.7 * 365)
    train_df_c.loc[i, 'FPull_4d07_365MPa_nosquare'] = round(f_pull_5d07_365MPa_nosquare, 1)

    f_pull_4d_310MPa = (np.pi/4) * np.square(4 * np.sqrt(t)) * (0.8 * 440)
    train_df_c.loc[i, 'FPull_4d_440MPa'] = round(f_pull_4d_310MPa, 1)
    f_pull_5d_310MPa = (np.pi/4) * np.square(5 * np.sqrt(t)) * (0.8 * 440)
    train_df_c.loc[i, 'FPull_5d_440MPa'] = round(f_pull_5d_310MPa, 1)

# About (0.8·σ)

Different sources mention different values for the 0,8. If you go with Tresca or von Mises criterion you get 0,5-0,57 which gets approximated to 0,6.  With 45 degree shear plane theroy the relationship is 1/√2 which is 0,707. It is also typical for 0,5 to be used. austenitic stainless steels can range up to ~ 0.8 this material is low carbon steel though and doesn't fall into this category.

# Check Dataset

In [13]:
train_df_c.head()

,Sample ID,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm),Material,PullTest (N),...,FPull_4d_365MPa,FPull_5d_365MPa,FPull_4d_440MPa,FPull_5d_440MPa,FPull_4d06_365MPa,FPull_4d07_365MPa,FPull_4d06_365MPa_nosquare,FPull_4d07_365MPa_nosquare,FPull_4d_sig_06_365MPa,FPull_4d_sig_07_365MPa
0,347,60,600,15,90.45,3403.28,0.621,0.620,AISI 1010 carbon steel,3062.9,...,1990.6,2275.0,2742.5,4285.1,1990.6,2275.0,1057.9,1234.2,1706.3,1990.6
1,111,80,800,0,115.49,3371.80,0.638,0.634,AISI 1010 carbon steel,2832.0,...,2035.6,2326.4,2804.4,4381.9,2035.6,2326.4,1106.2,1290.6,1744.8,2035.6
2,198,60,400,0,92.08,3113.48,0.630,0.622,AISI 1010 carbon steel,2665.1,...,1997.1,2282.4,2751.3,4299.0,1997.1,2282.4,1064.7,1242.2,1711.8,1997.1
3,195,60,400,0,92.05,3424.44,0.631,0.635,AISI 1010 carbon steel,2912.8,...,2026.0,2315.4,2791.1,4361.2,2026.0,2315.4,1095.8,1278.4,1736.5,2026.0
4,59,80,1000,0,97.24,4058.80,0.637,0.631,AISI 1010 carbon steel,2997.7,...,2026.0,2315.4,2791.1,4361.2,2026.0,2315.4,1095.8,1278.4,1736.5,2026.0


# Graph

In [9]:
x = train_df_c.index

fig = go.Figure()

groups = {
    "310MPa": {
        "cols": ["FPull_4d_310MPa", "FPull_5d_310MPa"],
        "colors": ["lightblue", "blue"]
    },
    "365MPa": {
        "cols": ["FPull_4d_365MPa", "FPull_5d_365MPa"],
        "colors": ["lightgreen", "green"]
    },
    "440MPa": {
        "cols": ["FPull_4d_440MPa", "FPull_5d_440MPa"],
        "colors": ["lightcoral", "red"]
    }
}

for info in groups.values():
    for col, col_color in zip(info["cols"], info["colors"]):
        fig.add_trace(
            go.Scatter(
                x=x,
                y=train_df_c[col],
                mode="lines+markers",
                name=col,
                line=dict(color=col_color, width=2),
                marker=dict(color=col_color)
            )
        )

fig.add_trace(
    go.Scatter(
        x=x,
        y=train_df_c["PullTest (N)"],
        mode="lines+markers",
        name="PullTest (N)",
        line=dict(color="grey", width=2, dash="dash"),
        marker=dict(color="grey", size=4)
    )
)

fig.add_trace(
    go.Scatter(
        x=x,
        y=train_df_c["NuggetDiameter (mm)"],
        mode="lines+markers",
        name="NuggetDiameter (mm)",
        line=dict(color="black", width=2),
        marker=dict(color="black", size=4),
        yaxis="y2"
    )
)

fig.update_layout(
    title="Pull Force and Nugget Diameter Over Different MPa Values and Nugget Diameter Recommendations",
    xaxis_title="Row Number",
    yaxis=dict(
        title="Force (N) or MPa",
        showgrid=True
    ),
    yaxis2=dict(
        title="Nugget Diameter (mm)",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    legend_title="Series",
    template="seaborn",
    legend=dict(
        x=1.1,        # push legend past the right edge
        y=1,
        xanchor="left",
        borderwidth=1
    ),
    margin=dict(r=200)  # give extra room on right for the legend
)

fig.show()

In [14]:
x = train_df_c.index

fig = go.Figure()

groups = {
    "sig": {
        "cols": ["FPull_4d_sig_06_365MPa", "FPull_4d_sig_07_365MPa"],
        "colors": ["lightblue", "blue"]
    },
    "365MPa": {
        "cols": ["FPull_4d_365MPa", "FPull_5d_365MPa"],
        "colors": ["lightgreen", "green"]
    },
    "nosquare": {
        "cols": ["FPull_4d06_365MPa_nosquare", "FPull_4d07_365MPa_nosquare"],
        "colors": ["lightcoral", "red"]
    }
}

for info in groups.values():
    for col, col_color in zip(info["cols"], info["colors"]):
        fig.add_trace(
            go.Scatter(
                x=x,
                y=train_df_c[col],
                mode="lines+markers",
                name=col,
                line=dict(color=col_color, width=2),
                marker=dict(color=col_color)
            )
        )

fig.add_trace(
    go.Scatter(
        x=x,
        y=train_df_c["PullTest (N)"],
        mode="lines+markers",
        name="PullTest (N)",
        line=dict(color="grey", width=2, dash="dash"),
        marker=dict(color="grey", size=4)
    )
)

fig.add_trace(
    go.Scatter(
        x=x,
        y=train_df_c["NuggetDiameter (mm)"],
        mode="lines+markers",
        name="NuggetDiameter (mm)",
        line=dict(color="black", width=2),
        marker=dict(color="black", size=4),
        yaxis="y2"
    )
)

fig.update_layout(
    title="Pull Force and Nugget Diameter Over Different MPa Values and Nugget Diameter Recommendations",
    xaxis_title="Row Number",
    yaxis=dict(
        title="Force (N) or MPa",
        showgrid=True
    ),
    yaxis2=dict(
        title="Nugget Diameter (mm)",
        overlaying="y",
        side="right",
        showgrid=False
    ),
    legend_title="Series",
    template="seaborn",
    legend=dict(
        x=1.1,        # push legend past the right edge
        y=1,
        xanchor="left",
        borderwidth=1
    ),
    margin=dict(r=200)  # give extra room on right for the legend
)

fig.show()